# Lab 3 — How Machine Learning Learns
## Notebook 03: Training Behavior and Generalization

**Estimated duration:** 55 minutes  
**Prerequisites:** Lab 2 (splitting datasets), Lab 3 Notebook 01 (model and loss), Notebook 02 (gradient descent)  
**Libraries:** NumPy, Pandas, Matplotlib (no scikit-learn model training)

In the previous notebooks you built predictions and minimized a loss. But **does a model that performs very well on its training data also perform well on new data?**

Here you will compare training and validation errors for models of different complexity. You will also investigate the effect of the number of training observations.

**Learning outcomes**

- Explain the difference between training and validation errors.
- Recognize **underfitting** and **overfitting** in model results and plots.
- Use a validation set to compare candidate models.
- Investigate how training-set size can change generalization.
- Explain why the validation results are estimates, not guarantees.

**How to work:** Execute the examples in order. Complete each cell marked **Exercise** or **Mini-challenge**, and answer the associated reflection questions in Markdown cells. The exercises contain no provided solution.

## 1. Load the data and create training and validation sets (9 min)

We use a new **synthetic, nonlinear dataset**. The measurements contain random noise, so a smooth curve cannot pass perfectly through every observation.

- `input_x`: one numerical input feature.
- `target_y`: a continuous value that the model should predict.

We deliberately use a **small training set** to make overfitting easier to observe. The remaining observations form the validation set.

A model's **training loss** measures how well it fits observations used to choose its parameters; **validation loss** measures how well it predicts observations *not used to fit those parameters*.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("data/nonlinear_data.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Cannot find data/nonlinear_data.csv. Open this notebook from lab03/ "
        "and keep the data/ folder beside the notebook."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
print(df.head())
print("Missing values:", df.isna().sum().to_dict())

In [ ]:
x_all = df["input_x"].to_numpy(dtype=float)
y_all = df["target_y"].to_numpy(dtype=float)

rng = np.random.default_rng(42)
indices = rng.permutation(len(df))
train_indices = indices[:22]
val_indices = indices[22:]

x_train, y_train = x_all[train_indices], y_all[train_indices]
x_val, y_val = x_all[val_indices], y_all[val_indices]

print(f"Training observations:   {len(x_train)}")
print(f"Validation observations: {len(x_val)}")

plt.figure(figsize=(8, 4.5))
plt.scatter(x_train, y_train, label="Training data", s=35)
plt.scatter(x_val, y_val, label="Validation data", s=35, alpha=0.65)
plt.xlabel("Input x")
plt.ylabel("Target y")
plt.title("Training and validation observations")
plt.legend()
plt.show()

**Exercise 1. Inspect the split (2–3 min).**

1. Calculate the fraction of the dataset used for training.
2. Check the smallest and largest `input_x` values in each subset.
3. Explain why a fixed random seed is helpful for a classroom experiment.

In [ ]:
# TODO (Exercise 1): compute the training fraction and x ranges.
# TODO: compare training and validation coverage of the x-axis.

## 2. Fit models with different complexity (12 min)

We will approximate the relationship between the input and target using a **polynomial model**:

\begin{equation*}
\hat{y} = a_0 + a_1 x + a_2 x^2 + \cdots + a_d x^d
\end{equation*}

Here `d` is the **degree** of the polynomial. A degree-1 polynomial is a straight line, while higher-degree polynomials can follow more complicated curves.

We use `np.polyfit()` to find coefficients that minimize squared errors on **training data only**, then `np.polyval()` to make predictions. This is a compact library implementation, not a new gradient-descent exercise.

- **Degree 1:** possibly too simple to capture a curved relationship.
- **Degree 2:** can represent a smooth quadratic curve.
- **Degree 12:** flexible enough to track noise in a small dataset.

In [ ]:
def fit_polynomial(x, y, degree):
    """Fit a polynomial to the supplied training observations."""
    return np.polyfit(x, y, deg=degree)


def predict_polynomial(coefficients, x):
    """Predict with coefficients returned by np.polyfit."""
    return np.polyval(coefficients, x)


def mean_squared_error(y_true, y_pred):
    """Mean squared prediction error, implemented with NumPy."""
    return np.mean((y_true - y_pred) ** 2)


candidate_degrees = [1, 2, 12]
models = {degree: fit_polynomial(x_train, y_train, degree)
          for degree in candidate_degrees}

x_plot = np.linspace(x_all.min(), x_all.max(), 300)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, degree in zip(axes, candidate_degrees):
    ax.scatter(x_train, y_train, s=23, label="Training", alpha=0.85)
    ax.scatter(x_val, y_val, s=16, label="Validation", alpha=0.35)
    ax.plot(x_plot, predict_polynomial(models[degree], x_plot),
            linewidth=2, label="Model")
    ax.set_title(f"Polynomial degree {degree}")
    ax.set_xlabel("Input x")
    ax.set_ylim(-1.5, 13)
axes[0].set_ylabel("Target y")
axes[0].legend(fontsize=8)
plt.suptitle("The effect of model complexity")
plt.tight_layout()
plt.show()

**Exercise 2. Interpret the fitted curves (3–4 min).**

1. Which model seems unable to capture the overall shape? Why?
2. Which curve appears to follow local irregularities rather than the overall trend?
3. Try changing `candidate_degrees` to `[1, 3, 9]`. Describe what changed.

*Note:* Changing the candidate list requires rerunning the model and plotting cell above.

In [ ]:
# TODO (Exercise 2): optionally fit degree 3 or 9 and plot its curve.
# Use fit_polynomial(...), predict_polynomial(...), and x_plot.

## 3. Compare training and validation losses (12 min)

Use **mean squared error (MSE)** as the loss:

\begin{equation*}
\operatorname{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
\end{equation*}

When a flexible model starts to fit the **random noise** in its training observations, its training loss can decrease even while validation loss increases. This is a typical indication of **overfitting**.

A model with both large training and validation losses may be **underfitting**. A useful model captures the underlying pattern and performs well on unseen observations.

In [ ]:
results = []
for degree in candidate_degrees:
    coefficients = models[degree]
    train_pred = predict_polynomial(coefficients, x_train)
    val_pred = predict_polynomial(coefficients, x_val)
    results.append({
        "degree": degree,
        "train_mse": mean_squared_error(y_train, train_pred),
        "validation_mse": mean_squared_error(y_val, val_pred),
    })
results_df = pd.DataFrame(results)
print(results_df.round(3).to_string(index=False))

plt.figure(figsize=(7, 4))
plt.plot(results_df["degree"], results_df["train_mse"], "o-", label="Training MSE")
plt.plot(results_df["degree"], results_df["validation_mse"], "s-", label="Validation MSE")
plt.xlabel("Polynomial degree")
plt.ylabel("Mean squared error")
plt.title("Training error vs. validation error")
plt.xticks(candidate_degrees)
plt.legend()
plt.show()

**Exercise 3. Select a model (3–4 min).**

1. Identify which model has the lowest **training** MSE.
2. Identify which model has the lowest **validation** MSE.
3. Why could those be different models?
4. Explain why selecting a model based only on its training performance can be misleading.

In [ ]:
# TODO (Exercise 3): identify the best degree using validation MSE.
# Hint: results_df.loc[results_df["validation_mse"].idxmin(), "degree"]

# TODO: print the degree and corresponding validation MSE.

## 4. Investigate the effect of training-set size (14 min)

Model flexibility is not the only factor that affects generalization. The **amount of training data** matters too.

To make a fairer comparison, we hold a fixed set of observations out for validation and fit the models using larger and larger subsets of the remaining training pool. Each fit uses its own training subset; no validation observation is used for parameter estimation.

We compare a sensible degree-2 polynomial with a much more flexible degree-12 polynomial.

In [ ]:
# A NEW split for this experiment, kept separate from the earlier one.
size_rng = np.random.default_rng(24)
ordered_indices = size_rng.permutation(len(x_all))
fixed_val_idx = ordered_indices[:25]
training_pool_idx = ordered_indices[25:]

x_fixed_val = x_all[fixed_val_idx]
y_fixed_val = y_all[fixed_val_idx]
training_sizes = [16, 24, 36, 48, 65]
size_results = []

for degree in [2, 12]:
    for n in training_sizes:
        subset_idx = training_pool_idx[:n]
        x_subset, y_subset = x_all[subset_idx], y_all[subset_idx]
        coefficients = fit_polynomial(x_subset, y_subset, degree)
        size_results.append({
            "degree": degree,
            "training_size": n,
            "train_mse": mean_squared_error(
                y_subset, predict_polynomial(coefficients, x_subset)),
            "validation_mse": mean_squared_error(
                y_fixed_val, predict_polynomial(coefficients, x_fixed_val)),
        })

size_df = pd.DataFrame(size_results)
print(size_df.round(3).to_string(index=False))

plt.figure(figsize=(8, 4.5))
for degree in [2, 12]:
    subset = size_df[size_df["degree"] == degree]
    plt.plot(subset["training_size"], subset["validation_mse"],
             "o-", label=f"Degree {degree}: validation MSE")
plt.xlabel("Number of training observations")
plt.ylabel("Validation MSE")
plt.title("More training data can help a flexible model generalize")
plt.legend()
plt.show()

**Exercise 4. Analyze learning behavior (4–5 min).**

1. For which degree does validation performance change most as the training set grows?
2. Does increasing the number of observations *always* reduce validation MSE at every step? Explain.
3. Compare the training MSE and validation MSE for degree 12 with 16 observations and 65 observations.
4. Why is the validation set kept the same during this experiment?

In [ ]:
# TODO (Exercise 4): extract and compare selected rows from size_df.
# Hint: size_df[(size_df["degree"] == 12) &
#               (size_df["training_size"].isin([16, 65]))]

## 5. Mini-challenge: Recommend a model (8 min)

Your task is to make and defend one practical decision: **Which polynomial degree should be used to predict new observations?**

1. Using the **original split** from Section 1 (`x_train`, `x_val`), fit degrees **1 through 12**.
2. Calculate their training and validation MSE.
3. Plot both loss curves against degree.
4. Select the model with the lowest validation MSE, not the lowest training MSE.
5. In 2–3 sentences, justify your decision and explain the remaining risk of overfitting.

**Important:** In a larger ML project, repeatedly adjusting models to this validation set can also lead to overfitting the validation set. A separate, untouched test set is needed for an unbiased final assessment. We will practise this more in Lab 4.

In [ ]:
# TODO (Mini-challenge): evaluate polynomial degrees 1 through 12.
# TODO: create a line plot of training and validation MSE.
# TODO: print the degree you recommend.

**Reflection — write your answers here**

- Which degree did you select, and why?
- What evidence of underfitting or overfitting did you observe?
- How could you evaluate your selected model more reliably in a real project?

## 6. Summary and preparation for Lab 4 (optional review)

You have now seen three different but connected ideas:

1. **A model fits parameters** to minimize loss on the training observations.
2. **Generalization requires evaluation on unseen observations.** A smaller training loss alone is not sufficient.
3. **Model complexity and training-set size interact.** A complex model can fit noise, especially with limited data.

For Lab 4, we will apply these principles to practical scikit-learn algorithms for **regression and classification**. Model selection will use validation data, with a separate test set reserved for final evaluation.

**Checkpoint:** Save the notebook with your completed exercises and written interpretations.